# Image-to-3D Benchmark — BATCH 2: TRELLIS + SPAR3D
Evaluates **TRELLIS** (Sparse Structured Latents + Rectified Flow -> FlexiCubes) and **SPAR3D** (Point-Aware Diffusion + Triplane DMTet + PBR) on the curated GSO benchmark under our locked evaluation protocol.

In [ ]:
# --- Error logging setup ---
import sys, traceback, datetime
from pathlib import Path
LOG_FILE = Path("/kaggle/working/outputs/benchmark_log.txt")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()
log_f = open(LOG_FILE, 'a')
sys.stdout = Tee(sys.stdout, log_f)
sys.stderr = Tee(sys.stderr, log_f)
print(f"=== BENCHMARK LOG START {datetime.datetime.now()} ===")
def log_exception(e):
    print(f"ERROR: {type(e).__name__}: {e}")
    traceback.print_exc()

In [ ]:
import os, subprocess, sys, glob, gc, json, tarfile
from pathlib import Path

# --- EGL must be set before ANY OpenGL/pyrender import (headless GPU rendering) ---
os.environ["PYOPENGL_PLATFORM"] = "egl"
# --- TRELLIS backends that work on Kaggle without compiling flash-attn ---
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
# --- HuggingFace authentication (for gated TRELLIS models) ---
# v28/v32 postmortem: the token WAS configured, but UserSecretsClient()
# raised BackendError: Unauthenticated (secrets service transient failure)
# -> check env first, then retry with longer backoff and catch all exceptions.
import time as _time
_hf_token = None
try:
    if os.environ.get("HF_TOKEN"):
        _hf_token = os.environ["HF_TOKEN"]
        print("HF_TOKEN found in environment ✅")
    else:
        from kaggle_secrets import UserSecretsClient
        for _attempt in (1, 2, 3, 4, 5):
            try:
                _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
                if _hf_token:
                    break
            except Exception as _se:
                _msg = str(_se)[:200]
                print(f"HF_TOKEN retrieval attempt {_attempt}/5 failed: "
                      f"{type(_se).__name__}: {_msg}")
                if "Unauthenticated" in _msg or "ConnectionError" in _msg:
                    _time.sleep(10)  # longer backoff for service issues
                else:
                    _time.sleep(5)
except ImportError:
    print("kaggle_secrets module not available - running outside Kaggle?")
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    print("HF_TOKEN set from Kaggle secrets ✅")
else:
    print("HF_TOKEN unavailable after 5 retries - TRELLIS will fail with 401")
    print("Check: Kaggle notebook → Settings → Secrets → 'HF_TOKEN' should have a value")


print("=== nvidia-smi ===")
try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else "!! nvidia-smi NOT FOUND - no GPU attached to this session")
except FileNotFoundError:
    print("!! nvidia-smi NOT FOUND - no GPU attached to this session")

import torch
print(f"torch {torch.__version__} (CUDA build: {torch.version.cuda})")
if not torch.cuda.is_available():
    raise RuntimeError(
        "torch.cuda.is_available() is False: this kernel is running on CPU.\n"
        "Fix: right sidebar > Session options > Accelerator > GPU T4 x2, then re-run all cells.\n"
        "(This is exactly what failed in the September 2026 run.)"
    )
props = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {props.name} | {props.total_memory/1e9:.1f} GB VRAM")

# ----------------- benchmark configuration -----------------
NUM_SAMPLES = 10   # 10-object validation run
GSO_DIR = "/kaggle/working/data/gso"
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)
print(f"Config: NUM_SAMPLES={NUM_SAMPLES} | GSO_DIR={GSO_DIR}")

# Pin numpy so later pip installs can NEVER upgrade it mid-process
# (a disk/memory numpy mismatch breaks numpy.testing imports via scipy)
import numpy
open("/tmp/constraints.txt", "w").write(f"numpy=={numpy.__version__}\n")
print(f"numpy pinned for all later installs: {numpy.__version__}")

In [ ]:
# [self-heal] session restarts wipe /tmp - recreate the numpy pin if missing
import os as _os, numpy as _np
if not _os.path.exists('/tmp/constraints.txt'):
    open('/tmp/constraints.txt', 'w').write(f"numpy=={_np.__version__}\n")
    print(f"[self-heal] recreated /tmp/constraints.txt (numpy=={_np.__version__})")
# --- shared dependencies (none of these may replace torch) ---
# NOTE: the Kaggle image ships rembg PRE-INSTALLED but WITHOUT onnxruntime, and pip
# skips dependency resolution for packages it considers "already satisfied" - so every
# transitive dep rembg needs at import/run time is declared explicitly here.
!pip install -q -c /tmp/constraints.txt trimesh scipy lpips rembg onnxruntime pooch opencv-python-headless pyrender pandas huggingface_hub "Pillow>=10.0"
import torch, subprocess, sys
assert torch.cuda.is_available(), "torch lost CUDA during installs - a package replaced the CUDA build"
# verify rembg imports cleanly in a FRESH interpreter (catches broken preinstalled deps)
r = subprocess.run([sys.executable, "-c", "import rembg"], capture_output=True, text=True)
if r.returncode != 0:
    print("rembg import failed once, retrying with explicit deps:", r.stderr.strip()[-400:])
    !pip install -q -c /tmp/constraints.txt --force-reinstall --no-deps onnxruntime
    r2 = subprocess.run([sys.executable, "-c", "import rembg"], capture_output=True, text=True)
    assert r2.returncode == 0, "rembg still broken: " + r2.stderr.strip()[-600:]
print("shared deps installed, torch CUDA intact, rembg imports cleanly")

# [warm-up] pre-download LPIPS VGG weights NOW, visibly, with the session's own python -
# compute_lpips() downloads lazily and a mid-run failure used to be swallowed (lpips = NaN forever)
_w = subprocess.run([sys.executable, "-c", "import lpips; lpips.LPIPS(net='vgg')"], capture_output=True, text=True)
assert _w.returncode == 0, "LPIPS warm-up failed: " + _w.stderr.strip()[-400:]
print("LPIPS(vgg) warmed up and cached")

from rembg import new_session
# [warm-up] pre-download ISNet (179 MB) NOW, visibly - the 972 MB BiRefNet
import time as _timer
print(f"[cell3] {datetime.datetime.now().strftime('%H:%M:%S')} Starting ISNet pre-warm (179 MB download)...")
_timer.sleep(0.1)
print(f"[cell3] {datetime.datetime.now().strftime('%H:%M:%S')} Creating isnet-general-use session...")
# model used to stall on the first TripoSR sample with zero progress output.
# ISNet (isnet-general-use) is a native rembg preset - same RGBA output,
# 9.5x smaller file. The benchmark code's BackgroundRemover class defaults
# to isnet-general-use now (see src/preprocessing/background_remover.py).
from PIL import Image as _PILImage
_synth = _PILImage.new('RGBA', (64, 64), (0, 255, 0, 255))
_sess = new_session('isnet-general-use')  # triggers download to /root/.u2net/
print(f'[cell3] {datetime.datetime.now().strftime("%H:%M:%S")} isnet-general-use session created and cached ✅')

# [gate] verify ALL critical imports in a fresh interpreter - never scroll past a failure silently
import subprocess as _gsp
_missing = []
for _m in ["trimesh", "scipy", "lpips", "rembg", "onnxruntime", "pyrender", "huggingface_hub"]:
    _g = _gsp.run([sys.executable, "-c", f"import {_m}"], capture_output=True, text=True)
    if _g.returncode != 0:
        _missing.append(_m)
        print(f"[gate] MISSING/BROKEN: {_m}: {_g.stderr.strip()[-200:]}")
if _missing:
    print(f"=== CELL 3 GATE: FAIL ({', '.join(_missing)}) === RE-RUN THIS CELL - do NOT continue to later cells")
else:
    print("=== CELL 3 GATE: PASS ===")

In [ ]:
# --- benchmark code: ALWAYS clone fresh from GitHub origin/main ---
import os, sys, glob, shutil, hashlib, subprocess
from pathlib import Path

CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
shutil.rmtree(CODE_DIR, ignore_errors=True)

_r = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", "main", "https://github.com/anubhavaanand/image-to-3d.git", CODE_DIR],
    capture_output=True, text=True
)
if _r.returncode != 0 or not os.path.exists(os.path.join(CODE_DIR, "src")):
    raise RuntimeError("GitHub clone failed:\n" + _r.stderr[-500:])

_sha = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"code cloned fresh from GitHub main @ {_sha[:12]}")

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

_f = hashlib.md5(open(os.path.join(CODE_DIR, "src/evaluation/benchmark_runner.py"), "rb").read()).hexdigest()[:10]
print("benchmark code ready at", CODE_DIR, f"(runner md5 {_f})")

from src.evaluation.benchmark_runner import BenchmarkRunner, load_gso_samples, bake_texture_to_vertices
from src.engines.triposr_engine import TriposrEngine
from src.engines.lgm_engine import LGMEngine
from src.engines.trellis_engine import TrellisEngine
from src.engines.spar3d_engine import Spar3dEngine
print("all 4 engines imported OK ✅")

In [ ]:
# --- ORDER GUARD: restores config globals if cell 2/4 did not run in this kernel
# (out-of-order execution or single-cell re-run after restart). Cheap + idempotent. ---
import os as _og_os, sys as _og_sys
from pathlib import Path as _og_Path
_og_os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
_og_os.environ.setdefault("ATTN_BACKEND", "xformers")
_og_os.environ.setdefault("SPCONV_ALGO", "native")
if "GSO_DIR" not in globals():
    GSO_DIR = "/kaggle/working/data/gso"
    print("[guard] GSO_DIR undefined -> restored default (cell 2 not run in this kernel?)")
if "NUM_SAMPLES" not in globals():
    NUM_SAMPLES = 10
    print("[guard] NUM_SAMPLES undefined -> default 10 (validation run)")
if "CODE_DIR" not in globals() or not _og_os.path.isdir(str(CODE_DIR)):
    _og_cand = "/kaggle/working/image-to-3d-benchmark"
    if _og_os.path.isdir(_og_cand):
        CODE_DIR = _og_cand
        print("[guard] CODE_DIR undefined -> restored:", CODE_DIR)
    else:
        raise RuntimeError("[guard] benchmark code not set up (cell 4 never ran). Run All from cell 1.")
if str(CODE_DIR) not in _og_sys.path:
    _og_sys.path.insert(0, str(CODE_DIR))
if "load_gso_samples" not in globals():
    from src.evaluation.benchmark_runner import load_gso_samples
if "bake_texture_to_vertices" not in globals():
    from src.evaluation.benchmark_runner import bake_texture_to_vertices
# --- GSO data: attach / download from verified mirror / handpicked list / self-render ---
import numpy as np, trimesh, zipfile, tempfile, shutil, random
from PIL import Image as PILImage

def count_valid(root):
    root = Path(root)
    if not root.exists():
        return 0
    n = 0
    for d in root.iterdir():
        if d.is_dir() and (list((d/"meshes").glob("*.obj")) or list((d/"meshes").glob("*.ply"))):
            n += 1
    return n

# 1) auto-detect a GSO-like dataset already attached as a Kaggle input
if count_valid(GSO_DIR) == 0:
    for cand in sorted(glob.glob("/kaggle/input/*")):
        n = count_valid(cand)
        if n > 20:
            GSO_DIR = cand
            print(f"using attached dataset: {cand} ({n} objects)")
            break

# 2) ingest from the official raw GSO mirror (1,046 per-object archives)
#    Handpick workflow: create handpicked.txt (one object name per line, from
#    gso_catalog.txt) and upload it via Kaggle's 'Upload > File' - it appears
#    in /kaggle/working/ or under /kaggle/input/. Without it, a seeded random
#    sample (seed 42) is used.
REPO = "suvadityamuk/google-scanned-objects-raw"
if count_valid(GSO_DIR) == 0:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    all_zips = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.endswith(".zip"))
    print(f"mirror has {len(all_zips)} object archives")

    wanted, handpicked = None, False
    for hp in ["/kaggle/working/handpicked.txt",
               os.path.join(CODE_DIR, "handpicked.txt"),
              ] + glob.glob("/kaggle/input/*/handpicked.txt"):
        if os.path.exists(hp):
            wanted = [l.strip() + ".zip" for l in open(hp)
                      if l.strip() and not l.strip().startswith("#")]
            wanted = [w for w in wanted if w in set(all_zips)]
            handpicked = True
            print(f"handpicked list: {len(wanted)} objects (from {hp})")
            break
    if wanted is None:
        random.Random(42).shuffle(all_zips)
        wanted = all_zips
        print(f"no handpicked.txt - using seeded random sample of {NUM_SAMPLES} (+spares)")

    def ingest(zip_name):
        obj_id = zip_name[:-4]
        try:
            zp = hf_hub_download(REPO, zip_name, repo_type="dataset",
                                 local_dir="/kaggle/working/gso_zips")
            with tempfile.TemporaryDirectory() as td:
                with zipfile.ZipFile(zp) as z:
                    z.extractall(td)
                hits = list(Path(td).rglob("*.obj")) + list(Path(td).rglob("*.ply"))
                if not hits:
                    return False
                mdir = Path(GSO_DIR) / obj_id / "meshes"
                mdir.mkdir(parents=True, exist_ok=True)
                for h in hits:
                    shutil.copy(h, mdir / h.name)
                for sib in hits[0].parent.iterdir():
                    if sib.suffix.lower() in (".mtl", ".png", ".jpg", ".jpeg"):
                        shutil.copy(sib, mdir / sib.name)
                # GSO layout: the MTL references texture.png, which lives in
                # materials/textures/ - NOT next to the .obj. Without it every
                # render is flat gray (bake_texture_to_vertices finds no image).
                for tex in Path(td).rglob("*.png"):
                    if not (mdir / tex.name).exists():
                        shutil.copy(tex, mdir / tex.name)
            return True
        except Exception as e:
            print("  skip", obj_id, str(e)[:80])
            return False

    target = min(len(wanted), NUM_SAMPLES + 10)
    ok = 0
    for zname in wanted:
        if ingest(zname):
            ok += 1
        if ok >= target:
            break
    print(f"ingested {ok} objects into {GSO_DIR}")

def render_cam(tm, az, res, white_bg):
    import pyrender
    scene = pyrender.Scene(ambient_light=np.array([0.65, 0.65, 0.65]))
    scene.add(pyrender.Mesh.from_trimesh(tm))  # keep vertex colors
    if white_bg:
        scene.bg_color = np.array([1.0, 1.0, 1.0, 1.0])
    azr = np.radians(az)
    pose = np.array([
        [ np.cos(azr), 0, np.sin(azr), 2.0*np.sin(azr)],
        [ 0,           1, 0,           0             ],
        [-np.sin(azr), 0, np.cos(azr), 2.0*np.cos(azr)],
        [ 0,           0, 0,           1             ]])
    scene.add(pyrender.PerspectiveCamera(yfov=np.pi/3.0), pose=pose)
    scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=1.8))
    ren = pyrender.OffscreenRenderer(res * 2, res * 2)
    col, _ = ren.render(scene, flags=pyrender.RenderFlags.SKIP_CULL_FACES)
    ren.delete()
    # 2x supersampling + downscale: sub-pixel triangles rasterize as a dot field at 1x
    col = np.array(PILImage.fromarray(col).resize((res, res), PILImage.LANCZOS))
    return col

# 3) objects with meshes but no images: render input view (000, white bg, 512)
#    and GT novel view (001, black bg 256 - SAME camera as the evaluator)
missing = 0
for d in sorted(Path(GSO_DIR).iterdir()):
    if not d.is_dir():
        continue
    mcands = list((d/"meshes").glob("*.obj")) + list((d/"meshes").glob("*.ply"))
    if not mcands or (d/"images"/"000.png").exists():
        continue
    try:
        tm = trimesh.load(str(mcands[0]), force="mesh")
        if tm is None or len(tm.faces) == 0:
            continue
        # sample the UV texture into vertex colors (to_color() = flat gray here)
        tm.visual = bake_texture_to_vertices(tm)
        pts = np.array(tm.vertices, dtype=np.float32)
        c = pts.mean(0)
        s = np.linalg.norm(pts - c, axis=1).max()
        tm.apply_translation(-c); tm.apply_scale(1.0/(s+1e-8))
        (d/"images").mkdir(parents=True, exist_ok=True)
        PILImage.fromarray(render_cam(tm, 0.0, 512, True)).save(d/"images"/"000.png")
        PILImage.fromarray(render_cam(tm, 45.0, 256, True)).save(d/"images"/"001.png")
        missing += 1
    except Exception as e:
        print("render failed:", d.name, str(e)[:100])
print(f"self-rendered images for {missing} objects")

n = count_valid(GSO_DIR)
print(f"GSO ready: {n} objects at {GSO_DIR}")
if n == 0:
    print("NO GSO data found - check the messages above, or attach a dataset with "
          "<id>/meshes/*.obj layout via Add Input and re-run this cell.")
else:
    samples = load_gso_samples(GSO_DIR, num_samples=NUM_SAMPLES)
    print(f"runner loaded {len(samples)} samples")

In [ ]:
# --- ORDER GUARD: restores config globals if cell 2/4 did not run in this kernel
# (out-of-order execution or single-cell re-run after restart). Cheap + idempotent. ---
import os as _og_os, sys as _og_sys
from pathlib import Path as _og_Path
_og_os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
_og_os.environ.setdefault("ATTN_BACKEND", "xformers")
_og_os.environ.setdefault("SPCONV_ALGO", "native")
if "GSO_DIR" not in globals():
    GSO_DIR = "/kaggle/working/data/gso"
    print("[guard] GSO_DIR undefined -> restored default (cell 2 not run in this kernel?)")
if "NUM_SAMPLES" not in globals():
    NUM_SAMPLES = 10
    print("[guard] NUM_SAMPLES undefined -> default 10 (validation run)")
if "CODE_DIR" not in globals() or not _og_os.path.isdir(str(CODE_DIR)):
    _og_cand = "/kaggle/working/image-to-3d-benchmark"
    if _og_os.path.isdir(_og_cand):
        CODE_DIR = _og_cand
        print("[guard] CODE_DIR undefined -> restored:", CODE_DIR)
    else:
        raise RuntimeError("[guard] benchmark code not set up (cell 4 never ran). Run All from cell 1.")
if str(CODE_DIR) not in _og_sys.path:
    _og_sys.path.insert(0, str(CODE_DIR))
if "TrellisEngine" not in globals():
    from src.engines.trellis_engine import TrellisEngine
# [self-heal] session restarts wipe /tmp - recreate the numpy pin if missing
import os as _os, numpy as _np
if not _os.path.exists('/tmp/constraints.txt'):
    open('/tmp/constraints.txt', 'w').write(f"numpy=={_np.__version__}\n")
    print(f"[self-heal] recreated /tmp/constraints.txt (numpy=={_np.__version__})")
# --- TRELLIS (structured latents + flow, ~30-90 s/object on T4) ---
# Complete install per TRELLIS setup.sh: submodules + diffoctreerast + utils3d + spconv-cu120 + xformers
import torch as _t

# 1. Install compatible spconv binary (only spconv-cu120 is officially supported for CUDA 12 per TRELLIS setup.sh)
# spconv wheel must match the PYTHON version (Kaggle image = py3.13, torch 2.11+cu128).
# spconv-cu126 2.3.8 is the ONLY official distribution with cp313 wheels (cu118/cu120 max out at py3.11);
# CUDA minor-version compatibility makes cu126 safe with torch cu128 and the 13.0 driver.
!pip install -q -c /tmp/constraints.txt spconv-cu126 || pip install -q -c /tmp/constraints.txt spconv-cu120
import subprocess as _sp2
_r2 = _sp2.run([sys.executable, '-c', 'import spconv.pytorch'], capture_output=True, text=True)
print('spconv import check:', 'OK' if _r2.returncode == 0 else 'FAILED: ' + _r2.stderr.strip()[-300:])

# 2. xformers (flash-attn won't compile on T4/P100 - Pascal arch)
try:
    import xformers
except ImportError:
    import os
    os.system('pip install -q -c /tmp/constraints.txt --no-deps xformers')

# 3. Core dependencies
!pip install -q -c /tmp/constraints.txt pytorch-lightning omegaconf easydict 'imageio[ffmpeg]' timm kornia open3d

# 4. nvdiffrast
!pip install -q -c /tmp/constraints.txt git+https://github.com/NVlabs/nvdiffrast.git

def _v53_ensure_checkout(repo_url, dest, label, pin, branch):
    """Clone `repo_url` to `dest` and leave it checked out at `pin`.

    Idempotent: an existing checkout is reused and re-pinned instead of
    re-cloned, so this cell can run any number of times in one session.
    """
    def _git(*args, check=True):
        _r = _sp2.run(['git', '-C', dest] + list(args),
                      capture_output=True, text=True)
        if _r.returncode != 0 and check:
            raise RuntimeError('git ' + ' '.join(args) + ' failed:\n'
                               + _r.stderr[-800:])
        return _r

    # a plain directory with no .git means an interrupted clone; git clone
    # refuses to reuse a non-empty directory, so clear it out first
    if _os.path.isdir(dest) and not _os.path.isdir(_os.path.join(dest, '.git')):
        print(f'[clone] [{label}] {dest} has no .git - removing and re-cloning')
        import shutil as _v53_sh
        _v53_sh.rmtree(dest, ignore_errors=True)
    if _os.path.isdir(_os.path.join(dest, '.git')):
        # reuse the existing checkout (shallow, so re-pinning is cheap)
        print(f'[clone] [{label}] reusing existing checkout at {dest}')
        if _git('cat-file', '-e', pin + '^{commit}', check=False).returncode == 0:
            _git('checkout', '-f', pin)
            _git('submodule', 'update', '--init', '--recursive', '--depth', '1')
            print(f'[clone] [{label}] at pin {pin[:12]} (cached, no fetch)')
            return
        try:
            _git('fetch', '--depth', '1', 'origin', pin)
            _git('checkout', '-f', pin)
            _git('submodule', 'update', '--init', '--recursive', '--depth', '1')
            print(f'[clone] [{label}] re-pinned to {pin[:12]}')
            return
        except RuntimeError:
            # a pinned commit can be refused when it is not an advertised
            # branch tip - fall back to the branch head and verify it
            print(f'[clone] [{label}] pin fetch refused, falling back to {branch}')
            _git('fetch', '--depth', '1', 'origin', branch)
            _git('checkout', '-f', branch)
            _head = _git('rev-parse', 'HEAD').stdout.strip()
            if _head != pin:
                raise RuntimeError(f'[clone] [{label}] expected pin {pin}, got {_head}')
            _git('submodule', 'update', '--init', '--recursive', '--depth', '1')
            print(f'[clone] [{label}] {branch} head is the pin ({_head[:12]})')
            return
    _c = _sp2.run(['git', 'clone', '--depth', '1', '--recurse-submodules',
                   repo_url, dest], capture_output=True, text=True)
    if _c.returncode != 0:
        raise RuntimeError('git clone ' + repo_url + ' failed:\n'
                           + _c.stderr[-800:])
    print(f'[clone] [{label}] cloned to {dest}')
    if _git('cat-file', '-e', pin + '^{commit}', check=False).returncode == 0:
        _git('checkout', '-f', pin)
        print(f'[clone] [{label}] pinned to {pin[:12]}')

# Cloned to a pinned commit, idempotently: a re-run must neither abort
# with 'already exists' nor fail silently, so every git call goes through
# subprocess and raises on failure (a ! magic only writes to stderr).
_v53_ensure_checkout('https://github.com/microsoft/TRELLIS.git',
                     '/kaggle/working/TRELLIS',
                     'TRELLIS', '442aa1e1afb9014e80681d3bf604e8d728a86ee7', 'main')
if '/kaggle/working/TRELLIS' not in sys.path:
    sys.path.insert(0, '/kaggle/working/TRELLIS')

# [unmask] TRELLIS's Pipeline.from_pretrained wraps the primary load path in a bare
# `except:` and falls back to a bogus repo_id (-> misleading 404). Patch it to print
# the REAL first-attempt error so failures are diagnosable.
_p8 = '/kaggle/working/TRELLIS/trellis/pipelines/base.py'
_s8 = open(_p8).read()
_old8 = '            except:\n                _models[k] = models.from_pretrained(v)'
_new8 = ('            except Exception as _e:\n'
         '                print(f"[TRELLIS-loader] {k}: primary path failed: {type(_e).__name__}: {_e}")\n'
         '                _models[k] = models.from_pretrained(v)')
if '[TRELLIS-loader]' in _s8:
    # already patched by an earlier run of this cell
    print('[TRELLIS-loader] base.py already patched')
elif _old8 in _s8:
    open(_p8, 'w').write(_s8.replace(_old8, _new8))
    print('[TRELLIS-loader] base.py unmasked')
else:
    raise AssertionError('base.py anchor not found - TRELLIS layout changed?')


# [kaolin shim] TRELLIS's ONLY kaolin usage is FlexiCubes' input-validation import
# (from kaolin.utils.testing import check_tensor) - verified by grepping the full
# repo + submodule. No kaolin wheel exists for torch 2.11/py3.13, so provide a
# faithful minimal stand-in via sys.modules instead of building kaolin from source.
import types as _t8types
if 'kaolin' not in sys.modules:
    _kaolin = _t8types.ModuleType('kaolin')
    _kaolin_utils = _t8types.ModuleType('kaolin.utils')
    _kaolin_testing = _t8types.ModuleType('kaolin.utils.testing')
    def _check_tensor(tensor, shape=None, throw=True, **_kw):
        import torch as _t
        ok = _t.is_tensor(tensor)
        if ok and shape is not None:
            want = tuple(None if s is None else int(s) for s in shape)
            ok = (len(want) == tensor.dim()) and all(
                s is None or a == s for s, a in zip(want, tuple(tensor.shape)))
        if ok:
            ok = bool(_t.isfinite(tensor).all())
        if not ok and throw:
            raise AssertionError(
                f"check_tensor failed: shape={tuple(tensor.shape)}, expected={shape}")
        return ok
    _kaolin_testing.check_tensor = _check_tensor
    _kaolin.utils = _kaolin_utils
    _kaolin.utils.testing = _kaolin_testing
    sys.modules['kaolin'] = _kaolin
    sys.modules['kaolin.utils'] = _kaolin_utils
    sys.modules['kaolin.utils.testing'] = _kaolin_testing
    print('[kaolin] shim installed (check_tensor) - no kaolin wheel needed')

# [cumm JIT headers] spconv's JIT op builds fail with "tensorview/core/all.h: No
# such file or directory": ccimport's generated build dir lacks cumm/include on
# its include path. Export CPATH so the JIT compiler finds the headers (the
# wheel DOES ship them) and clear stale build dirs so the next build re-runs.
import os as _co, glob as _c8g, shutil as _c8sh
_c8incs = sorted(_c8g.glob("/usr/local/lib/python3*/dist-packages/cumm/include")) + \
          sorted(_c8g.glob("/opt/conda/lib/python3*/site-packages/cumm/include"))
if _c8incs:
    _co.environ["CPATH"] = _c8incs[0] + (_co.pathsep + _co.environ["CPATH"] if _co.environ.get("CPATH") else "")
    print(f"[cumm] CPATH += {_c8incs[0]}")
    for _bd in _c8g.glob("/usr/local/lib/python3*/dist-packages/spconv/build") + \
               _c8g.glob("/usr/local/lib/python3*/dist-packages/cumm/build"):
        _c8sh.rmtree(_bd, ignore_errors=True)
        print(f"[cumm] cleared stale build dir: {_bd}")
else:
    print("[cumm] WARNING: cumm/include not found - TRELLIS JIT may fail on tensorview headers")

# 6. Install diffoctreerast as separate package (requires its own --recurse-submodules)
_v53_ensure_checkout('https://github.com/JeffreyXiang/diffoctreerast.git',
                     '/tmp/extensions/diffoctreerast',
                     'diffoctreerast', 'b09c20b84ec3aace4729e6e18a613112320eca3a', 'master')
!pip install -q /tmp/extensions/diffoctreerast

# 7. Install pinned utils3d (required by decoder_mesh.py -> SparseFeatures2Mesh)
!pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8

import torch
assert torch.cuda.is_available(), 'torch lost CUDA during installs'

# --- VRAM guard: auto-reduce TRELLIS steps for low-VRAM GPUs / validation runs ---
gpu_name = torch.cuda.get_device_properties(0).name
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 20 or NUM_SAMPLES <= 10:
    trellis_slat_steps = 8
    trellis_sparse_steps = 8
    print(f"GPU VRAM {vram_gb:.1f} GB or NUM_SAMPLES={NUM_SAMPLES} <= 10 -> reducing TRELLIS steps to 8/8")
else:
    trellis_slat_steps = 12
    trellis_sparse_steps = 12
    print(f"GPU VRAM {vram_gb:.1f} GB >= 20 GB -> using full TRELLIS steps 12/12")

# --- PROVEN single-GPU TRELLIS flow (10/10 on the trellis-smoke kernel) ---
# Plain TrellisEngine.load_model() puts the WHOLE pipeline on one GPU and OOMs
# at FlexiCubes decode on 15.6 GB T4s. Instead: local snapshot + selective
# placement (GS/RF decoders stay on CPU in mesh mode) + per-object staged
# offload of the flow models before decode + gaussian fallback on OOM.
import time, gc
import numpy as np
from huggingface_hub import snapshot_download
from trellis.pipelines import TrellisImageTo3DPipeline
from src.engines.base import Mesh3D, GaussianCloud, ReconstructionResult

print(f"[cell8] {datetime.datetime.now().strftime('%H:%M:%S')} fetching TRELLIS-image-large snapshot ...")
local_dir = snapshot_download("microsoft/TRELLIS-image-large")
print(f"[cell8] {datetime.datetime.now().strftime('%H:%M:%S')} loading pipeline from {local_dir}")
pipe = TrellisImageTo3DPipeline.from_pretrained(local_dir)

DEV = torch.device('cuda:0')
GS_NAME, RF_NAME = 'slat_decoder_gs', 'slat_decoder_rf'
for _name, _m in pipe.models.items():
    _m.to('cpu' if _name in (GS_NAME, RF_NAME) else DEV)
torch.cuda.empty_cache()
resid = {n: next(m.parameters()).device for n, m in pipe.models.items()}
print("model placement:", resid)

class MemHygieneTrellisEngine(TrellisEngine):
    """Pipeline pre-placed; @no_grad + staged offload + OOM->gaussian fallback."""

    def attach(self, pipeline, dev):
        self.pipeline = pipeline
        self.dev = dev

    @torch.no_grad()
    def predict(self, image):
        # The official pipeline.run() is decorated @torch.no_grad(); omitting it
        # retains activation graphs for every sampler step -> 14.2 GiB live at
        # decode time (the v4 OOM).
        if self.pipeline is None:
            self.load_model()
        p = self.pipeline
        if image.mode != "RGBA":
            image = image.convert("RGBA")

        torch.cuda.reset_peak_memory_stats(0)
        t0 = time.time()
        decode_mode = 'mesh'

        img = p.preprocess_image(image)
        cond = p.get_cond([img])
        torch.manual_seed(42)
        coords = p.sample_sparse_structure(cond, 1, {
            "steps": self.sparse_steps, "cfg_strength": self.cfg_strength})
        slat = p.sample_slat(cond, coords, {
            "steps": self.slat_steps, "cfg_strength": self.cfg_strength})
        del cond, coords
        gc.collect(); torch.cuda.empty_cache()

        # Staged offload: flow models + DINOv2 are dead weight during decode.
        # Move them to CPU so the ~5.3 GB FlexiCubes transient has the whole
        # GPU to itself.
        for _n in ('sparse_structure_flow_model', 'slat_flow_model', 'image_cond_model',
                   'sparse_structure_decoder'):
            if _n in p.models:
                p.models[_n].to('cpu')
        torch.cuda.empty_cache(); gc.collect()
        print("[TRELLIS] staged offload: flows+DINOv2 on CPU before decode")

        try:
            out = p.models['slat_decoder_mesh'](slat)
        except torch.OutOfMemoryError:
            print("[TRELLIS] mesh decode OOM -> gaussian fallback (GS decoder CPU->GPU)")
            torch.cuda.empty_cache(); gc.collect()
            p.models[GS_NAME].to(self.dev)
            with torch.cuda.device(self.dev):
                out = p.models[GS_NAME](slat)
            decode_mode = 'gaussian'
        latency = time.time() - t0
        print(f"[TRELLIS] inference done in {latency:.2f}s (decode_mode={decode_mode})")

        meta = {
            "engine": "TRELLIS",
            "output_format": decode_mode,
            "latency_s": round(latency, 3),
            "slat_steps": self.slat_steps,
            "sparse_steps": self.sparse_steps,
        }

        # restore the sampling models to GPU for the next object
        for _n in ('sparse_structure_flow_model', 'slat_flow_model', 'image_cond_model',
                   'sparse_structure_decoder'):
            if _n in p.models:
                p.models[_n].to(self.dev)
        torch.cuda.empty_cache()

        if decode_mode == 'mesh':
            raw = out[0]
            verts = raw.vertices.detach().cpu().numpy().astype(np.float32)
            faces = raw.faces.detach().cpu().numpy().astype(np.int32)
            if len(verts) == 0 or len(faces) == 0:
                raise RuntimeError("FlexiCubes produced an empty mesh (success=False?)")
            # keep TRELLIS per-vertex colors (vertex_attrs: 3 base + 3 normal);
            # renders/PSNR must see colored surfaces, not flat grey
            vcols = None
            _attrs = getattr(raw, "vertex_attrs", None)
            if _attrs is not None:
                try:
                    vcols = np.clip(
                        _attrs.detach().cpu().numpy().astype(np.float32)[:, :3], 0.0, 1.0)
                except Exception as _ce:
                    print(f"[TRELLIS] vertex color extraction failed (non-fatal): {_ce}")
            mesh = Mesh3D(vertices=verts, faces=faces, vertex_colors=vcols)
            meta.update({"vertex_count": len(mesh.vertices), "face_count": len(mesh.faces),
                         "has_vertex_colors": vcols is not None})
            torch.cuda.empty_cache(); gc.collect()
            return ReconstructionResult(mesh=mesh, metadata=meta)

        gs = out[0]
        gaussians = GaussianCloud(
            positions=gs.get_xyz.detach().cpu().numpy().astype(np.float32),
            scales=gs.get_scaling.detach().cpu().numpy().astype(np.float32),
            rotations=gs.get_rotation.detach().cpu().numpy().astype(np.float32),
            opacities=gs.get_opacity.detach().cpu().numpy().astype(np.float32),
            sh_coeffs=gs.get_features[:, 0, :3].detach().cpu().numpy().astype(np.float32),
        )
        meta["gaussian_count"] = len(gaussians.positions)
        # GS decoder back to CPU so a later mesh decode never finds it GPU-resident
        p.models[GS_NAME].to('cpu')
        torch.cuda.empty_cache(); gc.collect()
        return ReconstructionResult(gaussians=gaussians, metadata=meta)

    def get_engine_info(self):
        return {"name": "TRELLIS"}

try:
    tr_engine2 = MemHygieneTrellisEngine(device='cuda', output_format='mesh',
                                         slat_steps=trellis_slat_steps,
                                         sparse_steps=trellis_sparse_steps)
    tr_engine2.attach(pipe, DEV)
    runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[tr_engine2], num_samples=NUM_SAMPLES, output_dir='/kaggle/working/outputs/trellis')
    try:
        print(f"[cell8] {datetime.datetime.now().strftime('%H:%M:%S')} Starting engine run...")
        runner.run()
        print(f"[cell8] {datetime.datetime.now().strftime('%H:%M:%S')} Engine run COMPLETE ✅")
        # Save before the finally block deletes these names: saving after it
        # raises UnboundLocalError, which the handler reports as a skip.
        runner.save_results('/kaggle/working/outputs/trellis_results.json')
        display(runner.get_summary_df())
    finally:
        # Nuclear GPU cleanup after TRELLIS run
        del runner, tr_engine2
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.reset_accumulated_memory_stats()
except Exception as e:
    print(f"[TRELLIS] Skipped due to: {type(e).__name__}: {e}")
    # Don't mask with HF_TOKEN blame - real error is in the exception above

for _v in ['tr_engine2', 'runner']:
    if _v in dir():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

# [gate] make the TRELLIS outcome unmistakable
import json as _json8
try:
    _t8 = _json8.load(open('/kaggle/working/outputs/trellis_results.json'))
    print(f"=== CELL 8 GATE: {'PASS (' + str(len(_t8)) + ' result rows)' if _t8 else 'FAIL (empty - TRELLIS skipped, read the error above)'} ===")
except Exception as _e8:
    print(f"=== CELL 8 GATE: no results file: {_e8} ===")

In [ ]:
# --- SPAR3D (Point-Aware Diffusion + Triplane DMTet + PBR, ~1.8 s/object on T4) ---
import os, sys, gc, datetime, subprocess
from pathlib import Path
import torch

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ["SPAR3D_LOW_VRAM"] = "1"

if "CODE_DIR" not in globals():
    CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.engines.spar3d_engine import Spar3dEngine
from src.evaluation.benchmark_runner import BenchmarkRunner

# 1. Clone official Stability-AI/stable-point-aware-3d
SPAR3D_DIR = "/kaggle/working/stable-point-aware-3d"
if not os.path.isdir(os.path.join(SPAR3D_DIR, ".git")):
    shutil.rmtree(SPAR3D_DIR, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-point-aware-3d.git", SPAR3D_DIR],
        check=True,
    )
if SPAR3D_DIR not in sys.path:
    sys.path.insert(0, SPAR3D_DIR)

# 2. Install SPAR3D Python dependencies (protecting numpy and torch)
!pip install -q -c /tmp/constraints.txt einops jaxtyping omegaconf loralib ftfy regex
!pip install -q -c /tmp/constraints.txt --no-deps git+https://github.com/openai/CLIP.git
!pip install -q -c /tmp/constraints.txt --no-deps git+https://github.com/SunzeY/AlphaCLIP.git

# 3. Build SPAR3D's custom CUDA extensions (texture_baker and uv_unwrapper) with --no-build-isolation
print("[SPAR3D] Building texture_baker and uv_unwrapper CUDA extensions …")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/texture_baker"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/uv_unwrapper"],
    check=True,
)
print("[SPAR3D] texture_baker and uv_unwrapper built OK ✅")

assert torch.cuda.is_available(), "torch lost CUDA during SPAR3D installs"

try:
    sp_engine = Spar3dEngine(device="cuda", texture_resolution=1024, low_vram_mode=True)
    sp_engine.load_model()
    runner = BenchmarkRunner(
        dataset_path=GSO_DIR,
        engines=[sp_engine],
        num_samples=NUM_SAMPLES,
        output_dir="/kaggle/working/outputs/spar3d",
    )
    try:
        print(f"[SPAR3D] {datetime.datetime.now().strftime('%H:%M:%S')} Starting engine run...")
        runner.run()
        print(f"[SPAR3D] {datetime.datetime.now().strftime('%H:%M:%S')} Engine run COMPLETE ✅")
        runner.save_results("/kaggle/working/outputs/spar3d_results.json")
        display(runner.get_summary_df())
    finally:
        del runner, sp_engine
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
except Exception as e:
    print(f"[SPAR3D] Failed with {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# --- merge results, summary table, LaTeX export, Pareto plot ---
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for p in sorted(glob.glob("/kaggle/working/outputs/*_results.json")):
    try:
        rows += json.load(open(p))
    except Exception as e:
        print("skip", p, str(e)[:80])

if not rows:
    print("WARNING: No results found - run at least one engine cell above")
    rows = []
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/outputs/unified_results.csv", index=False)

num_cols = ["latency_s", "chamfer_distance", "f_score_0.1", "f_score_0.2",
            "f_score_0.5", "psnr", "lpips"]
if len(rows) == 0:
    print("No results - skipping summary table")
    cols = []
else:
    cols = [c for c in num_cols if c in df.columns]
if len(rows) == 0:
    print("No results - skipping summary table")
    cols = []
else:
    cols = [c for c in num_cols if c in df.columns]

if len(cols) == 0:
    print("=== UNIFIED BENCHMARK RESULTS ===")
    print("No numeric metrics in results. Check engine output above.")
    print(f"Available columns: {list(df.columns)}")
    df_summary_mean = None
    df_summary_std = None
else:
    summary = df.groupby("engine")[cols].agg(["mean", "std"]).round(4)
    print("=== UNIFIED BENCHMARK RESULTS (mean +/- std) ===")
    print(summary.to_string())

if len(cols) > 0:
    latex = summary.reset_index().to_latex(index=False, na_rep="--")
    with open("/kaggle/working/outputs/unified_table.tex", "w") as f:
        f.write(latex)
    print("saved: unified_results.csv, unified_table.tex")

    colors = {"TripoSR": "#1f77b4", "LGM": "#ff7f0e", "TRELLIS": "#2ca02c"}
    means = df.groupby("engine")[cols].mean()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for eng, row in means.iterrows():
        c = colors.get(eng, "grey")
        if "f_score_0.1" in row and not pd.isna(row.get("f_score_0.1")):
            axes[0].scatter(row["latency_s"], row["f_score_0.1"], s=180, c=c, label=eng, zorder=5)
        if "chamfer_distance" in row and not pd.isna(row.get("chamfer_distance")):
            axes[1].scatter(row["latency_s"], row["chamfer_distance"], s=180, c=c, label=eng, zorder=5)
    axes[0].set_xlabel("Latency (s)"); axes[0].set_ylabel("F-Score @ 0.1 (higher better)")
    axes[0].set_title("Speed vs F-Score"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel("Latency (s)"); axes[1].set_ylabel("Chamfer Distance (lower better)")
    axes[1].set_title("Speed vs Chamfer Distance"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/outputs/pareto_frontier.png", dpi=150)
    plt.show()
    print("saved: pareto_frontier.png - download /kaggle/working/outputs/ via the notebook Output tab")
else:
    # Still save CSV and empty table
    print("saved: unified_results.csv (no metrics)")
print("saved: pareto_frontier.png - download /kaggle/working/outputs/ via the notebook Output tab")

In [ ]:
# --- FINAL: strip /kaggle/working down to outputs/ so the saved version is a few MB ---
# Everything below is reproducible per-run (downloads, clones, extracted data).
# Only outputs/ (results JSONs, CSV, LaTeX, plot, benchmark log) needs persisting.
import shutil, os
from pathlib import Path

WORK = Path("/kaggle/working")
KEEP = {"outputs"}

def _dir_size(p):
    total = 0
    for root, _, files in os.walk(p):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return total

if WORK.exists():
    before = _dir_size(WORK)
    removed = []
    for item in sorted(WORK.iterdir()):
        if item.name in KEEP:
            continue
        try:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
            removed.append(item.name)
        except Exception as e:
            print(f"  could not remove {item.name}: {e}")
    after = _dir_size(WORK)
    print(f"cleanup: removed {len(removed)} items ({before/1e6:.0f} MB -> {after/1e6:.1f} MB)")
    print(f"kept: {sorted(KEEP)}")
    print("saved outputs:", sorted(p.name for p in (WORK / 'outputs').iterdir()) if (WORK / 'outputs').exists() else "NONE - check engine cells above")
else:
    print("no /kaggle/working - nothing to clean")